In [ ]:
%load_ext autoreload
%autoreload 2
import os
import pickle
from tqdm import tqdm

import torch
from esm.models.esmc import ESMC
from esm.tokenization import EsmSequenceTokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# Load model
head_out_dir = "../notebook_outs/09h_fit_esmc_contact_heads/esmc_600m/"
model = ESMC.from_pretrained("esmc_600m", use_flash_attn=False, init_contact_head=True, contact_head_weights_path=os.path.join(head_out_dir, "esmc_600m_contact_heads.pt")).to(device)
model.eval()
print("Successfully loaded model")
# Print number of parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {total_params:,}")

# Load tokenizer
tokenizer = EsmSequenceTokenizer()

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Successfully loaded model
Number of parameters: 575,037,641


In [4]:
# Alignments directory
target_data_file = "../../data/Figure2_heterooligomer_contact_prediction/target_data.pkl"
with open(target_data_file, "rb") as oFile:
    target_data_d = pickle.load(oFile)


In [ ]:
predicted_contacts_d = {}
for target in tqdm(target_data_d.keys()):
    msa_file = target_data_d[target]['msa_file']
    # Load alignment
    with open(msa_file, "r") as oFile:
        query_seq = oFile.readlines()[1].strip()
    len_seq = len(query_seq)
    # Predict contacts
    tokens = tokenizer.encode(query_seq, return_tensors="pt").to(device)
    with torch.no_grad():
        # Compute attention heads
        output = model(sequence_tokens=tokens, output_attentions=True, return_contacts=True)
        pred_contacts_a = output.contacts[0].float().cpu().numpy()
    predicted_contacts_d[target] = pred_contacts_a
with open("results/esmc/esmc_ppi_predicted_contacts.pkl", "wb") as oFile:
    pickle.dump(predicted_contacts_d, oFile)